In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


In [3]:
wiki_tool
    

WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Islam\\Documents\\llm-krish\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200))

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
loaded_docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(loaded_docs)


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
vectordb = FAISS.from_documents(documents, OpenAIEmbeddings())
retriver = vectordb.as_retriever()

In [6]:
retriver

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000226A62D6F90>, search_kwargs={})

In [7]:
# I have made tool for wikipedia and retrivers from web base loaders and vector stores.
# Now we need a retriver tool to retrive the data from the tools & then we will use the retrived data to answer the question.
# this is a customized tool I have made for retrivel and I made it a tool as agent dont know direct query retrieval but it have to choose a tool and then under the tool hood we have the actual retriver. that is the reason we keep the retiver inside a tool wrapper
from langchain_classic.tools.retriever import create_retriever_tool
retriver_tool = create_retriever_tool(retriever=retriver, name="langsmith_search", description="search for information about langsmith. For any information about langsmith, use this tool to search for it and get the answer. Always use this tool to search for information about langsmith. You must use this tool to search for information about langsmith. You should not answer any question about langsmith without using this tool to search for the information. Always use this tool to search for information about langsmith. You must use this tool to search for information about langsmith.")


In [8]:
retriver_tool.name

'langsmith_search'

In [9]:
# arxiv tool is used for fetching the paper onmline research papers. like we did for wikipedia it is an inbuilt tool in langchain
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

In [32]:
api_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper=api_wrapper)

In [37]:
from langchain_community.tools import DuckDuckGoSearchRun

# Create the tool
duckduckgo_tool = DuckDuckGoSearchRun()


In [38]:
# now combine all the tool in a list and then we will pass it to the agent
# tools = [wiki_tool, arxiv_tool, retriver_tool]
tools = [duckduckgo_tool, retriver_tool]

In [13]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Islam\\Documents\\llm-krish\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='search for information about langsmith. For any information about langsmith, use this tool to search for it and get the answer. Always use this tool to search for information about langsmith. You must use this tool to search for information about langsmith. You should not answer any question about langsmi

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model = "gpt-4.1-nano")  # nano


In [15]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("hwchase17/openai-functions-agent", dangerously_pull_public_prompt=True)

print(type(prompt))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [16]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [39]:
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm, tools, prompt)

In [40]:
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


In [43]:
agent_executor.invoke({"input": "give me a research paper on machine learning?"})



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': 'research paper on machine learning'}`


The paper introduced a new deep learning architecture known as the transformer, based on the attention mechanism proposed in 2014 by Bahdanau et al. Track papers/grants under trending research topics in real-time. COVID-19 (Biology & Health). Current research and grants related to the pandemic.The International Conference on Machine Learning (ICML) is one of the top machine learning conferences in the world. Machine learning research is great progress in many directions. This article summarizes these four directions and discusses some of them Current open issues. There are four directions (1) Improvement of classification accuracy by Methods of Learning Classmates Samples, (2) Augmenting... This research paper provides an overview of machine learning and artificial neural networks. It discusses various machine learning techniques like supervised learning, unsu

{'input': 'give me a research paper on machine learning?',
 'output': 'I found a research paper that provides an overview of machine learning and artificial neural networks. It discusses various techniques such as supervised learning, unsupervised learning, reinforcement learning, and deep learning. Would you like a detailed summary or the link to this paper?'}

In [ ]:
agent_executor.invoke({"input": "what is langsmith?"})